In [1]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     

    return pd.read_sql(sql, con)

In [2]:
def run_sql_file(path):                   # one sql file
    with open(path, 'r') as f:            # read only
        script = f.read()                 # whole file
    con.executescript(script)             # every statement
    con.commit()                          # save to the db
    print('ran', path)             
  

In [3]:
q("""select
    connection_type,
    count(*) as total_subscribers,
    round(
        count(*) * 100.0 / sum(count(*)) over (),
        2
    ) as percentage_share
from customers
group by connection_type;""")

,connection_type,total_subscribers,percentage_share
0,Postpaid,4851,22.05
1,Prepaid,17149,77.95


In [4]:
q("""select plan_name, plan_type, price_inr, count(*) as subs 
from customer_features 
group by plan_name, plan_type, price_inr 
order by subs desc 
limit 5;""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [5]:
q("""select
    c.circle_name,
    c.region,
    count(cu.customer_id) as total_subscribers
from customers cu
join circles c
    on cu.circle_id = c.circle_id
group by
    c.circle_name,
    c.region
order by total_subscribers desc
limit 5;""")

,circle_name,region,total_subscribers
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


In [6]:
q("""select
    age_band,
    count(*) as total_subscribers,
    round(
        count(*) * 100.0 / sum(count(*)) over (),
        2
    ) as percentage_share
from customers
group by age_band
order by age_band;""")

,age_band,total_subscribers,percentage_share
0,18-24,5268,23.95
1,25-34,7467,33.94
2,35-44,4824,21.93
3,45-54,2895,13.16
4,55+,1546,7.03


In [7]:
q("""select
    acquisition_channel,
    count(*) as subs,
    round(
        count(*) * 100.0 / sum(count(*)) over (),
        2
    ) as percentage_share
from customers
group by acquisition_channel
order by subs desc;""")

,acquisition_channel,subs,percentage_share
0,Retailer,9287,42.21
1,Airtel App,6513,29.60
2,Website,3081,14.00
3,Call Center,1770,8.05
4,Doorstep,1349,6.13


In [8]:
q("""select
    device_type,
    os,
    count(*) as subs
from customers
group by device_type, os
order by subs desc;""")

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


In [9]:
q("""select
    tenure_years,
    count(*) as subs,
    round(100.0 * count(*) / sum(count(*)) over (), 2) as perc
from customer_features
group by tenure_years
order by tenure_years;""")

,tenure_years,subs,perc
0,0,4610,20.95
1,1,6961,31.64
2,2,7055,32.07
3,3,3374,15.34


In [10]:
"""B1 · What is our average monthly revenue per subscriber,
 by the company's ARPU figure? — 

STEPS
1. Open customer_features.
2. Average stated_arpu.
3. Report it."""

q("""
SELECT AVG(stated_arpu) AS avg_arpu
FROM customer_features
""")

,avg_arpu
0,421.645818


In [11]:
# B2 · What is our average monthly revenue per subscriber, from the payments received?
# STEPS
# 1. Open customer_features again.
# 2. Average actual_arpu_6m, which Step 2 built from recharges.
# 3. Also total it, to get the monthly revenue of the whole base.

q("""
SELECT ROUND(AVG(actual_arpu_6m),2) AS avg_arpu,
       ROUND(SUM(actual_arpu_6m),2) AS total_revenue
FROM customer_features
""")

,avg_arpu,total_revenue
0,344.46,7578210.14


In [12]:
# B3 · By how much does the company's ARPU figure overstate the money received?
# STEPS
# 1. Average both columns in one query.
# 2. Average the gap.
# 3. Express the difference as a percentage of the actual total, not of the stated one.

q("""
SELECT ROUND(AVG(stated_arpu),2) AS avg_stated_arpu,
       ROUND(AVG(actual_arpu_6m),2) AS avg_actual_arpu,
       ROUND(AVG(arpu_gap),2) AS avg_gap,
       ROUND(100.0 * (SUM(stated_arpu) - 
       SUM(actual_arpu_6m)) / SUM(actual_arpu_6m), 2) AS gap_perc
FROM customer_features
""")

,avg_stated_arpu,avg_actual_arpu,avg_gap,gap_perc
0,421.65,344.46,77.18,22.41


In [13]:
# B4 · Is that gap the same for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Average both columns inside each group.
# 3. Read the two rows against each other rather than down the column.

q("""
SELECT connection_type,
       ROUND(AVG(stated_arpu),2) AS avg_stated_arpu,
       ROUND(AVG(actual_arpu_6m),2) AS avg_actual_arpu
FROM customer_features
GROUP BY connection_type
""")

,connection_type,avg_stated_arpu,avg_actual_arpu
0,Postpaid,959.65,970.42
1,Prepaid,269.46,167.40


In [14]:
#B5 · Which five plans bring in the most actual monthly revenue?

#STEPS
#1. Group by plan_name.
#2. Sum actual_arpu_6m for total monthly revenue and average it for revenue per subscriber.
#3. Sort by total, take five.

q("""select plan_name,
count(*) as subs,
sum(actual_arpu_6m) as total_revenue,
round(avg(actual_arpu_6m),2) as avg_revenue
from customer_features
group by plan_name
order by actual_arpu_6m desc
limit 5;""")

,plan_name,subs,total_revenue,avg_revenue
0,Postpaid Infinity 1499,1431,2171756.20,1517.65
1,Postpaid Family 999,1239,1249245.94,1008.27
2,Postpaid 749,946,718958.94,760.00
3,Postpaid 499,711,355720.25,500.31
4,Postpaid 399,524,211817.18,404.23


In [15]:
#B6 · Which five circles bring in the most actual monthly revenue?

#STEPS
#1. Group by circle_name.
#2. Sum actual_arpu_6m.
#3. Sort and take five.

q("""select circle_name,
count(*) as subs,
sum(actual_arpu_6m) as total_revenue,
round(avg(actual_arpu_6m),2) as avg_revenue
from customer_features
group by circle_name
order by total_revenue desc
limit 5;""")

,circle_name,subs,total_revenue,avg_revenue
0,Mumbai,1395,473288.16,339.27
1,Karnataka,1361,468514.95,344.24
2,Andhra Pradesh,1412,463369.80,328.17
3,North East,1318,460998.29,349.77
4,Maharashtra,1428,450032.30,315.15


In [16]:
#B7 · How much money did we receive from January to June 2025?

#STEPS
#1. This is a question about payments, not subscribers, so read recharges.
#2. Filter to the window.
#3. Count and sum.

q("""
SELECT COUNT(*) AS num_recharges,
SUM(amount_inr) AS total_revenue
FROM recharges
WHERE recharge_date BETWEEN '2025-01-01' AND '2025-06-30'
""")

,num_recharges,total_revenue
0,83693,45469268


In [17]:
#B8 · Do subscribers who have been with us longer pay more each month?

#STEPS
#1. Group by tenure_years.
#2. Average actual_arpu_6m.
#3. Sort by year

q("""
SELECT tenure_years,COUNT(*) AS subs,
ROUND(AVG(actual_arpu_6m),2) AS avg_revenue
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subs,avg_revenue
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


In [18]:
#C1 · How many payments of each type did we take, and from how many subscribers?

#STEPS
#1. Read recharges, where one row is one payment.
#2. Group by txn_type.
#3. Count rows and distinct subscribers in each group.

q("""select txn_type, count(distinct customer_id) as subs,
count(*) as payments
from recharges
group by txn_type;""")

,txn_type,subs,payments
0,Bill Payment,4851,108419
1,Recharge,17149,299670


In [19]:
#C2 · What is the average value of a payment? — deliberate failure 2

#STEPS
#1. Read recharges.
#2. Average amount_inr.

q("""
   SELECT ROUND(AVG(amount_inr), 2) AS avg_payment
   FROM recharges;""")

,avg_payment
0,532.59


In [20]:
#C3 · What is the average payment for a prepaid recharge, and for a postpaid bill?

#STEPS
#1. Same table, same average.
#2. Add GROUP BY txn_type.
#3. Add the minimum and maximum so the two ranges can be compared.

q("""
SELECT txn_type, round(avg(amount_inr), 2) as avg,
min(amount_inr) as min,
max(amount_inr) as max
FROM recharges
GROUP BY txn_type;""")

,txn_type,avg,min,max
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


In [21]:
#C4 · How many times did a prepaid and a postpaid subscriber pay in six months?

#STEPS
#1. Back to customer_features, because this is now a question about subscribers.
#2. Group by connection_type.
#3. Average recharge_count and recharge_total_6m.

q("""select connection_type, sum(recharge_count) as recharge_count,
round(avg(recharge_count), 2) as avg_rech_count,
sum(recharge_total_6m) as total_revenue,
round(avg(recharge_total_6m), 2) as avg_total
from customer_features
group by connection_type;""")

,connection_type,recharge_count,avg_rech_count,total_revenue,avg_total
0,Postpaid,26156,5.39,28244989,5822.51
1,Prepaid,57537,3.36,17224279,1004.39


In [22]:
#C5 · Which payment channels take the most payments?

#STEPS
#1. Read recharges.
#2. Group by channel.
#3. Count and share.

q("""select channel, count(*) as payment,
round(100.0 * count(*)/408090, 2)as perc
from recharges
group by channel
order by payment desc;""")

,channel,payment,perc
0,Website,111163,27.24
1,Airtel App,111083,27.22
2,USSD,75257,18.44
3,Retailer,74406,18.23
4,AutoPay,36180,8.87


In [23]:
#C6 · Do prepaid and postpaid subscribers pay through the same channels?

#STEPS
#1. Group by txn_type and channel.
#2. Use a window function to express each count as a share of its own txn_type, so the two groups can be compared fairly.

q("""select txn_type, channel, count(*) as n,
round(100.0 * count(*) / sum(count(*)) over (partition by txn_type), 2) as perc
from recharges
group by txn_type, channel
order by txn_type, n desc;""")

,txn_type,channel,n,perc
0,Bill Payment,Airtel App,36186,33.38
1,Bill Payment,AutoPay,36180,33.37
2,Bill Payment,Website,36053,33.25
3,Recharge,USSD,75257,25.11
4,Recharge,Website,75110,25.06
5,Recharge,Airtel App,74897,24.99
6,Recharge,Retailer,74406,24.83


In [24]:
#C7 · Does the number of payments a subscriber made go with whether they left?

#STEPS
#1. Group customer_features by recharge_count.
#2. Count subscribers and average churn in each group.
#3. Restrict to counts of eight or fewer, where the groups are large enough to read.

q("""select recharge_count, count(*) as subs,
round(100*avg(is_churned), 2) as churn
from customer_features
group by recharge_count
order by recharge_count;""")

,recharge_count,subs,churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63
9,9,30,86.67


In [25]:
#D1 · How much data did the network carry each month, and for how many subscribers?

#STEPS
#1. Read v_usage_month, where one row is one subscriber in one month.
#2. Group by month.
#3. Sum the data and convert megabytes to terabytes.
#4. Also count the distinct subscribers who appear, because that number is going to matter.

q("""
SELECT month,
ROUND(SUM(total_mb) / 1024 / 1024, 2) AS data_tb,
COUNT(DISTINCT customer_id) AS subs
FROM v_usage_month
GROUP BY month
""")

,month,data_tb,subs
0,2025-01,337.97,21302
1,2025-02,307.32,21162
2,2025-03,332.51,20366
3,2025-04,310.27,19320
4,2025-05,363.10,18356
5,2025-06,338.95,17326


In [26]:
#  D2 · What is a subscriber's average daily data use?
# STEPS
# 1. Read usage_daily.
# 2. Average data_mb.

q("""
SELECT ROUND(AVG(data_mb),2) AS avg_daily_data
FROM usage_daily
""")

,avg_daily_data
0,1572.87


In [27]:
# D3 · What is the average data per subscriber, per active day and per observed day?
# STEPS
# 1. Read customer_features, where Step 2 already computed both.
# 2. Average each.
# 3. Divide one by the other.
q("""SELECT ROUND(AVG(mb_per_observed_day), 2) AS avg_obsrv_day,
    ROUND(AVG(mb_per_active_day), 2) AS avg_active_day
    FROM customer_features;""")

,avg_obsrv_day,avg_active_day
0,524.05,1404.44


In [28]:
#D4 · Does data use per day change with the day of the week?

#STEPS
#1. Read usage_daily.
#2. Use strftime('%w', usage_date) to pull the weekday out.
#3. Group by it.
#4. Count rows and average the data.

q("""
SELECT strftime('%w', usage_date) AS weekday,
    COUNT(*) AS num_rows,
    ROUND(AVG(data_mb), 2) AS avg_data_mb
FROM usage_daily
GROUP BY weekday;
""")

,weekday,num_rows,avg_data_mb
0,0,190368,1578.06
1,1,189914,1572.94
2,2,183823,1569.55
3,3,191088,1564.48
4,4,190764,1580.00
5,5,190542,1569.51
6,6,190240,1575.45


In [29]:
#D5 · How many subscribers used the network in each month?
#STEPS
#1. Read v_usage_month.
#2. Count distinct subscribers per month.
#3. Read the trend rather than any single number.
q("""
SELECT month, COUNT(DISTINCT customer_id) AS subscribers
FROM v_usage_month
GROUP BY month
ORDER BY month
""")

,month,subscribers
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [30]:
#D6 · Did voice minutes and SMS per day change from January to June? STEPS 1. Read usage_daily. 2. Bucket by month with strftime. 3. Average voice_minutes #and sms_count.
q("""
SELECT strftime('%Y-%m', usage_date) AS month,
	ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
	ROUND(AVG(sms_count), 2) AS avg_sms_count
FROM usage_daily
WHERE usage_date >= '2025-01-01'
  AND usage_date < '2025-07-01'
GROUP BY month
ORDER BY month
""")

,month,avg_voice_minutes,avg_sms_count
0,2025-01,55.56,0.75
1,2025-02,55.84,0.75
2,2025-03,56.08,0.75
3,2025-04,56.49,0.76
4,2025-05,56.70,0.76
5,2025-06,57.13,0.77


In [31]:
#D7 · How much data did an active subscriber use each month, and on how many days?
#STEPS
#1. Read v_usage_month.
#2. Group by month.
#3. Average total_mb per subscriber and convert to gigabytes.
#4. Also average usage_days, to see whether people are appearing more often or using more when they 
#appear.
q("""
SELECT month,
    ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_subscriber,
    ROUND(AVG(usage_days), 2) AS avg_usage_days
FROM v_usage_month
GROUP BY month
ORDER BY month
""")

,month,avg_gb_per_subscriber,avg_usage_days
0,2025-01,16.25,11.39
1,2025-02,14.87,10.31
2,2025-03,16.72,11.52
3,2025-04,16.44,11.25
4,2025-05,20.26,11.73
5,2025-06,20.03,11.46


In [32]:
# D8 · Among subscribers who never left, did data per active day still rise in May?
# Hold the cohort constant by keeping only subscribers who never churned.
q("""
SELECT
    v.month,
    COUNT(DISTINCT v.customer_id) AS subscribers,
    ROUND(AVG(1.0 * v.total_mb / NULLIF(v.usage_days, 0)), 2) AS avg_mb_per_active_day
FROM v_usage_month AS v
JOIN customer_features AS c
    ON c.customer_id = v.customer_id
WHERE c.is_churned = 0
GROUP BY v.month
ORDER BY v.month
""")

,month,subscribers,avg_mb_per_active_day
0,2025-01,16293,1410.58
1,2025-02,16588,1422.15
2,2025-03,16640,1418.06
3,2025-04,16648,1411.76
4,2025-05,16667,1659.45
5,2025-06,16660,1662.24


In [33]:
q("""
  
with new_total as (select count(*)as total from customer_features)
select * from new_total;

""")

,total
0,22000


In [34]:
q("""
    WITH ranked AS (
        SELECT
            NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile,
            mb_per_active_day
        FROM customer_features
        WHERE mb_per_active_day > 0
    )
    SELECT
        quartile,
        COUNT(*) AS subscribers,
        MIN(mb_per_active_day) AS min_mb_per_active_day,
        MAX(mb_per_active_day) AS max_mb_per_active_day
    FROM ranked
    GROUP BY quartile
    ORDER BY quartile
""")

,quartile,subscribers,min_mb_per_active_day,max_mb_per_active_day
0,1,5499,25.46,657.32
1,2,5499,657.35,1092.83
2,3,5499,1092.84,1571.87
3,4,5499,1572.12,9719.60


In [35]:
q("""
    SELECT
        c.connection_type,
        c.device_type,
        ROUND(AVG(u.voice_minutes), 2) AS avg_voice_minutes_per_active_day,
        ROUND(AVG(u.data_mb), 2) AS avg_mb_per_active_day,
        COUNT(*) AS active_days
    FROM usage_daily AS u
    JOIN customer_features AS c
        ON c.customer_id = u.customer_id
    GROUP BY c.connection_type, c.device_type
    ORDER BY avg_voice_minutes_per_active_day DESC
""")

,connection_type,device_type,avg_voice_minutes_per_active_day,avg_mb_per_active_day,active_days
0,Postpaid,Feature Phone,56.71,591.87,42674
1,Postpaid,Smartphone,56.33,3724.20,259745
2,Prepaid,Smartphone,56.24,1209.17,883352
3,Prepaid,Feature Phone,56.17,184.87,140968


In [36]:
q("""
select plan_name,monthly_gb,
round(avg(total_data_mb)/1024,2) as avg_gb
from customer_features
group by plan_name,monthly_gb
order by monthly_gb;""")

,plan_name,monthly_gb,avg_gb
0,Prepaid Lite 99,15.0,17.31
1,Prepaid Smart 155,30.0,34.28
2,Prepaid Value 209,30.0,35.33
3,Postpaid 399,40.0,47.13
4,Prepaid Max 479,45.0,53.06
5,Prepaid Popular 299,45.0,53.59
6,Prepaid Annual 1799,60.0,71.20
7,Prepaid Quarterly 719,60.0,70.67
8,Postpaid 499,75.0,92.82
9,Prepaid Data 349,75.0,90.76


In [37]:
q("""
  select distinct(m_score)from customer_features;""")

,m_score
0,3
1,1
2,4
3,2


In [38]:
q("""
SELECT m_score, COUNT(*) AS subs,
ROUND(AVG(recharge_total_6m),0) AS spended,
ROUND(AVG(mb_per_active_day),0) AS mb_per_day,
ROUND(100*AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY m_score
ORDER BY m_score;
""")

,m_score,subs,spended,mb_per_day,churn
0,1,5500,167.0,973.0,54.82
1,2,5500,945.0,779.0,19.96
2,3,5500,1678.0,1117.0,11.24
3,4,5500,5477.0,2750.0,9.00


In [39]:
q("""
SELECT 
    ROUND(AVG(total_sms * 1.0 / NULLIF(usage_days, 0)), 2) AS sms_per_day,
    SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_used,
    COUNT(*) AS subs
FROM customer_features
WHERE usage_days > 0;
""")

,sms_per_day,never_used,subs
0,0.6,654,21996


In [40]:
q("""
 select is_roaming, COUNT(*) AS subs,
 round(100*count(*)/1326739,2)as perc,
 count(distinct customer_id) as subs
 from usage_daily
group by is_roaming;
""")

,is_roaming,subs,perc,subs
0,0,1287143,97.0,21994
1,1,39596,2.0,16233


In [41]:
# F2 · Do subscribers use more or less on roaming days than at home?
# STEPS
# 1. Same table, same grouping.
# 2. Average data_mb and voice_minutes in each group.
# 3. Read the two rows against each other.

q("""
SELECT is_roaming,
       ROUND(AVG(voice_minutes), 2) AS voice_min,
       ROUND(AVG(data_mb), 2) AS data
FROM usage_daily
GROUP BY is_roaming;

""")

,is_roaming,voice_min,data
0,0,56.27,1572.95
1,1,56.09,1570.05


In [42]:
q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    ROUND(AVG(roaming_days), 2) AS avg_roaming_days,
    ROUND(100.0 * AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0.0 END), 2)
        AS pct_ever_roamed
FROM customer_features
GROUP BY connection_type
""")

,connection_type,subscribers,avg_roaming_days,pct_ever_roamed
0,Postpaid,4851,1.84,74.83
1,Prepaid,17149,1.79,73.49


In [43]:
q("""
SELECT
    circle_name,
    ROUND(100*SUM(roaming_days) * 1.0 / NULLIF(SUM(usage_days), 0), 2) AS roaming_per_usage_day,
    COUNT(*) AS subs
FROM customer_features
GROUP BY circle_name
ORDER BY roaming_per_usage_day DESC
LIMIT 5;
""")

,circle_name,roaming_per_usage_day,subs
0,Odisha,3.07,696
1,Haryana,3.07,1041
2,Delhi,3.07,735
3,Karnataka,3.05,1361
4,Assam,3.05,1301


In [44]:
q("""
SELECT CASE WHEN roaming_days=0 THEN '0'
WHEN roaming_days <= 2 THEN '0-2'
WHEN roaming_days <= 5 THEN '3-5'
ELSE '6+'
END AS band,
COUNT(*) AS subs,
ROUND(100.0*AVG(is_Churned),2) AS churn
FROM customer_features
WHERE usage_days>0
GROUP BY band
ORDER BY band;
""")

,band,subs,churn
0,0,5763,44.61
1,0-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


In [45]:
# G1 · What do subscribers complain about most?
# STEPS
# 1. Read complaints, where one row is one ticket.
# 2. Group by category.
# 3. Count and share.
# 4. Sort biggest first.
q("""
SELECT category, COUNT(*) AS tickets,
ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM complaints),2) AS perc
FROM complaints
GROUP BY category
ORDER BY tickets DESC;
""")

,category,tickets,perc
0,Slow Data,8035,29.94
1,Dropped Call,6365,23.71
2,Network Coverage,4885,18.20
3,Billing,3738,13.93
4,Recharge Issue,2679,9.98
5,SIM/KYC,1139,4.24


In [46]:
# G2 · Does the complaint channel change how often a ticket is resolved?
q("""
SELECT
    channel,
    COUNT(*) AS tickets,
    ROUND(100.0 * AVG(is_resolved), 2) AS resolved_pct
FROM complaints
GROUP BY channel
ORDER BY tickets DESC
""")

,channel,tickets,resolved_pct
0,Store,6799,70.33
1,Call Center,6728,69.72
2,Twitter/X,6698,70.36
3,Airtel App,6616,70.04


In [47]:
# G3 · How many tickets are still open?
q("""
SELECT
    CASE WHEN is_resolved = 0 THEN 'Open' ELSE 'Resolved' END AS status,
    COUNT(*) AS tickets,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 2) AS share_pct
FROM complaints
GROUP BY is_resolved
""")

,status,tickets,share_pct
0,Open,8021,29.88
1,Resolved,18820,70.12


In [48]:
q("""SELECT category,COUNT(*) AS tickets,
ROUND(100.0*SUM(is_resolved)/COUNT(*),2)AS resolved_perc,
ROUND(AVG(resolution_days),2) AS avg_resolved_days
FROM complaints
GROUP BY category
ORDER BY resolved_perc;
""")

,category,tickets,resolved_perc,avg_resolved_days
0,SIM/KYC,1139,68.57,7.65
1,Network Coverage,4885,69.52,7.60
2,Dropped Call,6365,69.71,7.46
3,Slow Data,8035,69.97,7.55
4,Billing,3738,70.87,7.50
5,Recharge Issue,2679,72.23,7.55


In [49]:

# 2 G5 · How many days does a resolved ticket take to close?
# STEPS
# 1. Filter to resolved tickets only.
# 2. Average, minimum and maximum resolution_days.
# 3. Say explicitly what population that is measured on.
q("""
SELECT
    COUNT(*) AS resolved_tickets,
    ROUND(AVG(resolution_days), 2) AS avg_resolution_days,
    MIN(resolution_days) AS min_resolution_days,
    MAX(resolution_days) AS max_resolution_days
FROM complaints
WHERE is_resolved = 1
""")

,resolved_tickets,avg_resolution_days,min_resolution_days,max_resolution_days
0,18820,7.54,1.0,14.0


In [50]:
q("""
SELECT
    CASE
        WHEN complaint_count = 0 THEN '0'
        WHEN complaint_count = 1 THEN '1'
        ELSE '2+'
    END AS complaint_band,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY complaint_band
ORDER BY CASE complaint_band
    WHEN '0' THEN 0
    WHEN '1' THEN 1
    ELSE 2
END
""")

,complaint_band,subscribers
0,0,6631
1,1,7854
2,2+,7515


In [51]:
# · Which five circles raise the most complaints per subscriber?
# STEPS
# 1. Group by circle_name.
# 2. Divide total tickets by subscribers so circles of different sizes compare.
# 3. Sort and take five
q("""
SELECT circle_name,
    COUNT(*) AS subs,
    ROUND(1.0 * SUM(complaint_count) / COUNT(*), 2) AS perc
    FROM customer_features
    GROUP BY circle_name
    ORDER BY perc DESC
    LIMIT 5;
""")

,circle_name,subs,perc
0,Tamil Nadu,731,1.29
1,Bihar,887,1.29
2,Punjab,558,1.27
3,Kolkata,572,1.27
4,Delhi,735,1.27


In [52]:
q("""
SELECT strftime('%Y', activation_date) AS joined_year,
    COUNT(*) AS subscribers
FROM customer_features
WHERE is_churned = 0
GROUP BY joined_year
ORDER BY joined_year
""")

,joined_year,subscribers
0,2022,5548
1,2023,5293
2,2024,5152
3,2025,781


In [53]:
# H2 · What share of each joining year has left?
q("""
SELECT
    strftime('%Y', activation_date) AS joined_year,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
GROUP BY joined_year
ORDER BY joined_year
""")

,joined_year,subscribers,churned_subscribers,churn_pct
0,2022,6922,1374,19.85
1,2023,6925,1632,23.57
2,2024,7053,1901,26.95
3,2025,1100,319,29.00


In [54]:
# H3 · How does churn change with each completed year on the network?
q("""
SELECT
    tenure_years,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subscribers,churned_subscribers,churn_pct
0,0,4610,1328,28.81
1,1,6961,1725,24.78
2,2,7055,1541,21.84
3,3,3374,632,18.73


In [55]:
# H4 · How long had a leaver been with us when they left?
# STEPS
# 1. Filter to churned subscribers.
# 2. Average, minimum and maximum days_to_churn, which Step 2 built from the two dates.
q("""
SELECT
    COUNT(*) AS Leavers,
    AVG(days_to_churn) AS avg_days,
    MIN(days_to_churn) AS min_days,
    MAX(days_to_churn) AS max_days
FROM customer_features
WHERE is_churned = 1
""")

,Leavers,avg_days,min_days,max_days
0,5226,567.183123,10,1259


In [56]:
# Do longer-standing subscribers pay more?
q("""
SELECT
    tenure_years,
    COUNT(*) AS subscribers,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_payment,
    ROUND(AVG(recharge_total_6m), 2) AS avg_recharge_total_6m
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subscribers,avg_monthly_payment,avg_recharge_total_6m
0,0,4610,329.10,1974.63
1,1,6961,342.72,2056.30
2,2,7055,350.13,2100.76
3,3,3374,357.22,2143.29


In [58]:
q("""
SELECT
    COUNT(*) AS total_sub,
    SUM(is_churned) AS sub_lost,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
""")

,total_sub,sub_lost,churn_rate_pct
0,22000,5226,23.75


In [59]:
# I2 · What is the churn rate for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Count, sum and average churn.
q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY connection_type
""")

,connection_type,subscribers,churned,churn_rate_pct
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


In [61]:
# I3 · Which six plans have the highest churn rate?
# STEPS
# 1. Group by plan_name, carrying plan_type.
# 2. Count and average churn.
# 3. Sort worst first, take six.
q("""
SELECT plan_name, plan_type,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY plan_name, plan_type
ORDER BY churn_rate_pct DESC
LIMIT 6
""")

,plan_name,plan_type,subscribers,churn_rate_pct
0,Prepaid Lite 99,Prepaid,996,31.12
1,Prepaid Smart 155,Prepaid,1137,30.17
2,Prepaid Entertainment 399,Prepaid,2142,26.24
3,Prepaid Quarterly 719,Prepaid,2561,25.85
4,Prepaid Value 209,Prepaid,1437,25.26
5,Prepaid Annual 1799,Prepaid,2901,25.13


In [63]:
q("""
SELECT circle_name, region,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn_rate_pct DESC
""")

,circle_name,region,subscribers,churn_rate_pct
0,Himachal Pradesh,North,1247,33.44
1,Maharashtra,West,1428,32.91
2,Andhra Pradesh,South,1412,28.90
3,Mumbai,West,1395,28.10
4,Uttar Pradesh (East),North,554,27.26
5,Haryana,North,1041,25.46
6,Jammu & Kashmir,North,933,24.87
7,Gujarat,West,1210,24.63
8,Assam,East,1301,23.14
9,Uttar Pradesh (West),North,1081,22.94


In [65]:
# I5 · Do smartphone users leave at a different rate from feature-phone users?
q("""
SELECT
    device_type,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY device_type
ORDER BY device_type
""")

,device_type,subscribers,churned,churn_rate_pct
0,Feature Phone,3029,719,23.74
1,Smartphone,18971,4507,23.76


In [69]:
# I6 · Does churn differ by acquisition channel?
q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY acquisition_channel
ORDER BY churn_rate_pct DESC
""")

,acquisition_channel,subscribers,churned,churn_rate_pct
0,Call Center,1770,436,24.63
1,Airtel App,6513,1576,24.20
2,Website,3081,743,24.12
3,Doorstep,1349,321,23.80
4,Retailer,9287,2150,23.15


In [71]:
# I7 · Does churn differ by age band?
q("""
SELECT
    age_band,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY age_band
ORDER BY age_band
""")

,age_band,subscribers,churned,churn_rate_pct
0,18-24,5268,1246,23.65
1,25-34,7467,1770,23.70
2,35-44,4824,1144,23.71
3,45-54,2895,680,23.49
4,55+,1546,386,24.97


In [73]:
q("""
SELECT
    CASE
        WHEN complaint_count = 0 THEN 'No complaints'
        WHEN unresolved_count = 0 THEN 'Complaints, none open'
        ELSE 'At least one open complaint'
    END AS complaint_status,
    COUNT(*) AS sub,
    SUM(is_churned) AS churned_sub,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY complaint_status
ORDER BY CASE complaint_status
    WHEN 'No complaints' THEN 1
    WHEN 'Complaints, none open' THEN 2
    ELSE 3
END
""")

,complaint_status,sub,churned_sub,churn_rate_pct
0,No complaints,6631,1328,20.03
1,"Complaints, none open",8689,2064,23.75
2,At least one open complaint,6680,1834,27.46


In [75]:
q("""
SELECT
    unresolved_count,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY unresolved_count
ORDER BY unresolved_count
""")


,unresolved_count,subscribers,churned,churn_rate_pct
0,0,15320,3392,22.14
1,1,5515,1438,26.07
2,2,1018,337,33.10
3,3,120,45,37.50
4,4,25,12,48.00
5,5,2,2,100.00


In [76]:

# K1 · Churn rate in each recency quartile
q("""
SELECT
    r_score,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY r_score
ORDER BY r_score
""")

,r_score,subscribers,churned,churn_rate_pct
0,1,5500,5032,91.49
1,2,5500,194,3.53
2,3,5500,0,0.00
3,4,5500,0,0.00


In [77]:
# K2 · What is the churn rate in each frequency quartile?
# Check the average recharge_count and churn rate for each f_score.
q("""
SELECT
    f_score,
    COUNT(*) AS subscribers,
    ROUND(AVG(recharge_count), 2) AS avg_recharge_count,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY f_score
ORDER BY f_score
""")

,f_score,subscribers,avg_recharge_count,churn_rate_pct
0,1,5500,0.33,45.44
1,2,5500,2.51,38.67
2,3,5500,5.46,7.76
3,4,5500,6.92,3.15


In [79]:
# K3 · Spend and churn rate in each spend quartile
q("""
SELECT
    m_score,
    COUNT(*) AS subscribers,
    ROUND(AVG(recharge_total_6m), 2) AS avg_spend,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY m_score
ORDER BY m_score
""")

,m_score,subscribers,avg_spend,churn_rate_pct
0,1,5500,166.77,54.82
1,2,5500,945.26,19.96
2,3,5500,1678.08,11.24
3,4,5500,5477.03,9.00


In [80]:
# L1 · What are the churn rate and the monthly revenue of each region?
q("""
SELECT
    region,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct,
    ROUND(SUM(actual_arpu_6m), 2) AS monthly_revenue
FROM customer_features
GROUP BY region
ORDER BY churn_rate_pct DESC
""")

,region,subscribers,churn_rate_pct,monthly_revenue
0,West,4665,27.85,1571780.02
1,North,7231,23.73,2477493.21
2,East,5459,21.91,1893199.66
3,South,4645,21.85,1635737.25
